In [1]:
!pip install chess
import chess
import chess.pgn
from io import StringIO #converts games to strings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
!pip install requests #needed this to call api's
import requests
from tensorflow import keras
from tensorflow.keras import layers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 65.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=527dca79e5ad57db37dc1640a21109bec8ee7242465ef5b8db0d2028944184b9
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [2]:
##stockfish for future implementation
url = "https://chess-api.com/v1"

data = {
    "fen": "r1bqkbnr/pppp1ppp/2n5/4p3/1b1P4/2N2N2/PPP1PPPP/R1BQKB1R w KQkq - 2 4", #get data for this board position
    "depth": 10, #analyse data to this extent
}
response = requests.post(url, json=data)
print(response.status_code)
print(response.text)


200
{
  "text": "Move d4 → e5 (dxe5): [-4.24]. Black is winning. Depth 10.",
  "captured": "p",
  "promotion": false,
  "isCapture": true,
  "isPromotion": false,
  "isCastling": false,
  "fen": "r1bqkbnr/pppp1ppp/2n5/4p3/1b1P4/2N2N2/PPP1PPPP/R1BQKB1R w KQkq - 2 4",
  "type": "bestmove",
  "depth": 10,
  "move": "d4e5",
  "eval": -4.24,
  "centipawns": "-424",
  "mate": null,
  "continuationArr": [
    "b4c3",
    "b2c3",
    "d7d6",
    "c1g5"
  ],
  "debug": "info depth 10 seldepth 16 multipv 1 score cp -424 nodes 101365 nps 25341250 hashfull 27 tbhits 0 time 4 pv d4e5 b4c3 b2c3 d7d6 c1g5",
  "winChance": 17.34742469694894,
  "taskId": "rdlxl6bk6",
  "turn": "w",
  "color": "w",
  "piece": "p",
  "from": "d4",
  "to": "e5",
  "san": "dxe5",
  "flags": "c",
  "lan": "d4e5",
  "fromNumeric": "44",
  "toNumeric": "55",
  "continuation": [
    {
      "from": "b4",
      "to": "c3",
      "fromNumeric": "24",
      "toNumeric": "33"
    },
    {
      "from": "b2",
      "to": "c3",
    

In [3]:
def board_to_tensor(board): #if you have a board ready, you can send it through this function to convert it to a tensor
  tensor = np.zeros((12, 8, 8), dtype=np.float32) #3D array made, zeros makes the whole 3d array full of zeros
  piece_map = board.piece_map() #looks at each square
  for square, piece in piece_map.items():
      row, col = divmod(square, 8)
      channel = piece.piece_type - 1
      if piece.colour == chess.BLACK:
        channel+=6
      tensor[channel, row, col] = 1.0
  return torch.from_numpy(tensor)

def fetch_games(username, year, month):
    url = f"https://api.chess.com/pub/player/{username}/games/{year}/{month:02d}"
    headers = {"User-Agent": "chess-analytics/1.0"}
    response = requests.get(url, headers=headers)
    games = response.json().get("games", [])
    return [game["pgn"] for game in games if game.get("pgn")]
#pgns = fetch_games("tanishk2850", 2025, 6)
#print(f"Fetched {len(pgns)} games")
def fetch_games_lichess(username):
    url = f"https://lichess.org/api/games/user/{username}"
    headers = {
        "Accept": "application/x-chess-pgn",
        "User-Agent": "chess-analytics/1.0"
    }
    response = requests.get(url, headers=headers, params={"max": 20})
    raw = response.text
    games = raw.strip().split("\n\n\n")
    return [g.strip() for g in games if g.strip()]

#now that we have the tensor of the game
#we have to give it features to look at based on the tensor

# feature 1: blunder with the move

def is_hanging_piece(board, move):

  square = move.to_square
  piece_color = not board.turn
  opponent_color = board.turn
  is_attacked = board.is_attacked_by(opponent_color, square)
  is_defended = board.is_attacked_by(piece_color, square)

  return is_attacked and not is_defended

#feature 2: blunder after the move
def failed_to_protect(previous_board, current_board, color):
    result = False
    for square in chess.SQUARES:
        piece = previous_board.piece_at(square)
        if piece and piece.color == color:
            was_hanging = (
                previous_board.is_attacked_by(not color, square) and
                not previous_board.is_attacked_by(color, square)
            )
            still_hanging = (
                current_board.is_attacked_by(not color, square) and
                not current_board.is_attacked_by(color, square)
            )
            if was_hanging and still_hanging:
                result = True
                break
    return result

#feature 3: underdeveloped?
def is_underdeveloped(board, move_number, color):
      if move_number < 10:
          return False

      if color == chess.WHITE:
          knight_squares = [chess.B1, chess.G1]
          bishop_squares = [chess.C1, chess.F1]
          pawn_squares = [chess.C2, chess.D2, chess.E2, chess.F2]
      else:
          knight_squares = [chess.B8, chess.G8]
          bishop_squares = [chess.C8, chess.F8]
          pawn_squares = [chess.C7, chess.D7, chess.E7, chess.F7]

      bishops_home = sum(1 for sq in bishop_squares
                   if board.piece_at(sq) and board.piece_at(sq).piece_type == chess.BISHOP)

      knights_home = sum(1 for sq in knight_squares
                        if board.piece_at(sq) and board.piece_at(sq).piece_type == chess.KNIGHT)

      pawns_home = sum(1 for sq in pawn_squares
                      if board.piece_at(sq) and board.piece_at(sq).piece_type == chess.PAWN)

      return knights_home == 2 or bishops_home == 2 or pawns_home >= 3

#feature 4:
def weaker_middle(board, move_number, color):
      if move_number <= 10 or move_number >= 25:
          return False #dont check for the first 10 moves, check after that >>
      centre_squares = [
        chess.B3, chess.C3, chess.D3, chess.E3, chess.F3, chess.G3,
        chess.B4, chess.C4, chess.D4, chess.E4, chess.F4, chess.G4,
        chess.B5, chess.C5, chess.D5, chess.E5, chess.F5, chess.G5,
        chess.B6, chess.C6, chess.D6, chess.E6, chess.F6, chess.G6
      ]
      white_count = sum(1 for sq in centre_squares
                      if board.piece_at(sq) and board.piece_at(sq).color == chess.WHITE)
      black_count = sum(1 for sq in centre_squares
                      if board.piece_at(sq) and board.piece_at(sq).color == chess.BLACK)

      if color == chess.WHITE:
        return white_count < black_count
      else:
        return black_count < white_count


In [4]:

def analyse_game(pgn_string, username):
  game = chess.pgn.read_game(StringIO(pgn_string))
  board = game.board()
  white_name = game.headers["White"]
  black_name = game.headers["Black"]

  if username == white_name:
      player_colour = "white"
  else:
      player_colour = "black"

  opponent_colour = "black" if player_colour == "white" else "white"

  move_analysis = []
  positions = []

  for i, move in enumerate(game.mainline_moves()):
      board.push(move)
      #is piece hanging

      failed_to_protect_result = False
      if i > 0:
        failed_to_protect_result = failed_to_protect(positions[i-1], board, not board.turn)

      move_data = {
          "move_number": board.fullmove_number,
          "player": "white" if not board.turn else "black",
          "move": move.uci(),
          "is_hanging": is_hanging_piece(board, move),
          "failed_to_protect": failed_to_protect_result,
          "is_underdeveloped": is_underdeveloped(board, board.fullmove_number, not board.turn),
          "weaker_middle": weaker_middle(board, board.fullmove_number, not board.turn)

      }
      move_analysis.append(move_data)
      positions.append(board.copy())

  df = pd.DataFrame(move_analysis)
  hanging = df.groupby("player")["is_hanging"].sum()
  failed = df.groupby("player")["failed_to_protect"].sum()
  underdeveloped = df.groupby("player")["is_underdeveloped"].sum()
  centre = df.groupby("player")["weaker_middle"].sum()


  player_won = game.headers["Result"] == "1-0" if player_colour == "white" else game.headers["Result"] == "0-1"

  return {
        "hanging": hanging[player_colour],
        "failed_to_protect": failed[player_colour],
        "underdeveloped": underdeveloped[player_colour],
        "weaker_middle": centre[player_colour],
        "hanging": hanging[player_colour],
        "failed_to_protect": failed[player_colour],
        "underdeveloped": underdeveloped[player_colour],
        "weaker_middle": centre[player_colour],
        "opponent_hanging": hanging[opponent_colour],
        "opponent_failed": failed[opponent_colour],
        "opponent_underdeveloped": underdeveloped[opponent_colour],
        "opponent_weaker_middle": centre[opponent_colour],
        "player_won": player_won,
        "result": game.headers["Result"]
    }


In [ ]:
results = []
source = input("Which site do you want to get from. Chess.com or Lichess?")
if source == "Chess" or source == "Chess.com" or source == "chess" or source == "chess.com":
  username = input("Enter Chess.com username: ")
  year = int(input("Enter year (e.g. 2025): "))
  month = int(input("Enter month (e.g. 6): "))

  pgns = fetch_games(username, year, month)
  print(f"Fetched {len(pgns)} games")
  for pgn in pgns:
      try:
          result = analyse_game(pgn, username)
          results.append(result)
      except Exception as e:
          print(f"Skipped a game: {e}")
elif input == "Lichess" or "lichess":
  username = input("Enter Lichess username: ")
  year = int(input("Enter year (e.g. 2025): "))
  month = int(input("Enter month (e.g. 6): "))

  pgns = fetch_games_lichess(username)
  print(f"Fetched {len(pgns)} games")
  for pgn in pgns:
      try:
          result = analyse_game(pgn, username)
          results.append(result)
      except Exception as e:
          print(f"Skipped a game: {e}")

print(f"Successfully analysed {len(results)} games")

In [7]:
results = []

# validate source
while True:
    source = input("Which site do you want to get from. Chess.com or Lichess? ").strip().lower()
    if source in ["chess", "chess.com", "lichess"]:
        break
    print("Invalid input. Please enter 'Chess.com' or 'Lichess'.")

if source in ["chess", "chess.com"]:
    # username validation
    while True:
        username = input("Enter Chess.com username: ").strip()
        if username:
            break
        print("Username cannot be empty.")

    # year validation
    while True:
        try:
            year = int(input("Enter year (e.g. 2025): "))
            break
        except ValueError:
            print("Invalid year. Please enter a number like 2025.")

    # month validation
    while True:
        try:
            month = int(input("Enter month (e.g. 6): "))
            if 1 <= month <= 12:
                break
            else:
                print("Month must be between 1 and 12.")
        except ValueError:
            print("Invalid month. Please enter a number between 1 and 12.")

    pgns = fetch_games(username, year, month)
    print(f"Fetched {len(pgns)} games")
    for pgn in pgns:
        try:
            result = analyse_game(pgn, username)
            results.append(result)
        except Exception as e:
            print(f"Skipped a game: {e}")

elif source == "lichess":
    # username validation
    while True:
        username = input("Enter Lichess username: ").strip()
        if username:
            break
        print("Username cannot be empty.")

    # year validation
    while True:
        try:
            year = int(input("Enter year (e.g. 2025): "))
            break
        except ValueError:
            print("Invalid year. Please enter a number like 2025.")

    # month validation
    while True:
        try:
            month = int(input("Enter month (e.g. 6): "))
            if 1 <= month <= 12:
                break
            else:
                print("Month must be between 1 and 12.")
        except ValueError:
            print("Invalid month. Please enter a number between 1 and 12.")

    pgns = fetch_games_lichess(username)
    print(f"Fetched {len(pgns)} games")
    for pgn in pgns:
        try:
            result = analyse_game(pgn, username)
            results.append(result)
        except Exception as e:
            print(f"Skipped a game: {e}")

print(f"Successfully analysed {len(results)} games")

Which site do you want to get from. Chess.com or Lichess? chess
Enter Chess.com username: kryceus
Enter year (e.g. 2025): 2024
Enter month (e.g. 6): 11
Fetched 29 games
Successfully analysed 29 games


In [8]:
avg_hanging = sum(r["hanging"] for r in results) / len(results)
avg_failed = sum(r["failed_to_protect"] for r in results) / len(results)
avg_underdeveloped = sum(r["underdeveloped"] for r in results) / len(results)
avg_centre = sum(r["weaker_middle"] for r in results) / len(results)

print("=" * 45)
print(f"{'WEAKNESS REPORT':^45}")
print(f"{'Player: ' + username:^45}")
print(f"{'Games analysed: ' + str(len(results)):^45}")
print("=" * 45)
print(f"{'Feature':<25} {'Avg per game':>12}")
print("-" * 45)
print(f"{'Hanging pieces':<25} {avg_hanging:>12.2f}")
print(f"{'Failed to protect':<25} {avg_failed:>12.2f}")
print(f"{'Underdeveloped':<25} {avg_underdeveloped:>12.2f}")
print(f"{'Weaker centre':<25} {avg_centre:>12.2f}")
print("=" * 45)

correlation1 = 0
correlation2 = 0
correlation3 = 0
correlation4 = 0

games_worse_hanging = [r for r in results if r["hanging"] > r["opponent_hanging"]]
losses = [r for r in games_worse_hanging if not r["player_won"]]

if len(games_worse_hanging) > 0:
    correlation1 = len(losses) / len(games_worse_hanging)
    #print(f"Hanging pieces: lost {correlation1:.0%} of games where worse than opponent ({len(games_worse_hanging)} games)")

games_weaker_protector = [r for r in results if r["failed_to_protect"] > r["opponent_failed"]]
losses = [r for r in games_weaker_protector if not r["player_won"]]

if len(games_weaker_protector) > 0:
    correlation2 = len(losses) / len(games_weaker_protector)
    #print(f"Failed to protect pieces: lost {correlation2:.0%} of games where worse than opponent ({len(games_weaker_protector)} games)")

games_underdeveloped = [r for r in results if r["underdeveloped"] > r["opponent_underdeveloped"]]
losses = [r for r in games_underdeveloped if not r["player_won"]]

if len(games_underdeveloped) > 0:
    correlation3 = len(losses) / len(games_underdeveloped)
    #print(f"Underdeveloped: lost {correlation3:.0%} of games where worse than opponent ({len(games_underdeveloped)} games)")

games_weaker_middle = [r for r in results if r["weaker_middle"] > r["opponent_weaker_middle"]]
losses = [r for r in games_weaker_middle if not r["player_won"]]

if len(games_weaker_middle) > 0:
    correlation4 = len(losses) / len(games_weaker_middle)
    #print(f"Weaker middle: lost {correlation4:.0%} of games where worse than opponent ({len(games_weaker_middle)} games)")


weaknesses = {
    "Hanging pieces": correlation1,
    "Failed to protect": correlation2,
    "Underdeveloped": correlation3,
    "Weaker middle": correlation4
}

ranked = sorted(weaknesses.items(), key=lambda x: x[1], reverse=True)

print(f"\n=== TOP WEAKNESSES FOR {username} ===")
for i, (weakness, correlation) in enumerate(ranked, 1):
    print(f"{i}. {weakness}: {correlation:.0%} loss rate when worse than opponent")

               WEAKNESS REPORT               
               Player: kryceus               
             Games analysed: 29              
Feature                   Avg per game
---------------------------------------------
Hanging pieces                    3.17
Failed to protect                 6.66
Underdeveloped                    0.69
Weaker centre                     6.41

=== TOP WEAKNESSES FOR kryceus ===
1. Hanging pieces: 87% loss rate when worse than opponent
2. Failed to protect: 73% loss rate when worse than opponent
3. Underdeveloped: 67% loss rate when worse than opponent
4. Weaker middle: 67% loss rate when worse than opponent
